# 04 - SVM method

**Aim:** implement the practical SVM protocol for signal-peptide detection, with feature extraction and parameter selection inside nested cross-validation.

We used the practical instructions in `04 - Methods - SVM.pdf` to plan this section. We will reuse the representative proteins and folds from 02b, build composition, hydropathy, charge and propensity features, and select the SVM settings using training data only. The analysis in 02c provides biological context, and 03 provides our loading and reporting conventions.

**Materials**
- `04 - Methods - SVM.pdf` (slides 22–26 and 30–41; practical instructions on slide 41).
- `04_feature_extraction_hydrophobicity.ipynb` (setup, A.1–A.5 and B.4).
- `04_SVM-sklearn_revised.ipynb` (sections 1–3 and 7).
- `03_von_heijne_method.ipynb` (loading in 1.1; metrics and mean ± SE in 5.4–5.6).

- External feature rationale: [feature notes](../docs/04_feature_candidates.md).

**Status:** implementation is in progress; remaining tasks are marked pending.

## Work distribution and handoffs

We defined five consecutive blocks so each person can complete their contribution and pass the cumulative notebook to the next. Person 1 handles shared preparation; Person 5 handles final integration. Everyone can read the materials in advance, but implementation follows **Person 1 → Person 2 → Person 3 → Person 4 → Person 5**.

| Person | Scientific sections | Receives | Passes on |
| --- | --- | --- | --- |
| 1 | 0–1: data and common definitions | Prepared FASTA/TSV and practical materials. | To Person 2: `data`, reserved `benchmarking_data`, paths, fixed hydropathy settings and the training-only `get_n` helper. |
| 2 | 2: feature extraction | Person 1's data and definitions. | To Person 3: composition/profile functions, `extract_features(proteins, n)`, feature names and catalog. |
| 3 | 3: RF ranking and SVM helpers | Person 2's extractors and feature order. | To Person 4: ranking, coupled parameter-grid, scaler/SVM and evaluation helpers. |
| 4 | 4–5: inner selection and outer evaluation | Person 3's helpers and all preceding contributions. | To Person 5: five external-fold metrics, predictions, selected settings/features and inner-search scores. |
| 5 | 6: reporting and final integration | Person 4's results and the cumulative notebook. | To the group: integrated notebook, result tables, figures and observations. |

We kept names and interfaces common across the blocks:

| Object | Interface |
| --- | --- |
| `data` | Training proteins with `accession`, `sequence`, `label`, `split`, `cv_fold` and original metadata. |
| `get_n(train_data)` | Integer N-terminal length derived only from positive SP lengths in the supplied training subset. |
| `extract_features(proteins, n)` | Numeric DataFrame indexed by accession, with the same ordered feature columns for every subset. |
| RF ranking | Column indices ordered by importance, plus names and importances from the current training fit. |
| Search candidates | Valid tuples `(k, C, gamma)`, with gamma calculated for that candidate's k. |
| `inner_scores` | `test_fold`, `validation_fold`, `n`, `k`, `C`, `gamma`, `mcc`. |
| `cv_per_round` | `test_fold`, `n`, `selected_k`, `C`, `gamma`, `precision`, `recall`, `accuracy`, `mcc`, `f1`. |
| `cv_predictions` | `accession`, `test_fold`, `label`, `prediction`. |
| `selected_features` | `test_fold`, `feature`, `rank`, `importance`, after the outer-training refit. |

All fitted quantities belong to their current training subset. The handoffs contain definitions and reusable functions first, then actual results after Person 4 completes CV. Each person adds the explanation for their own block before handing it on.

## 0. Data loading

We kept paths relative to `notebooks/` and selected the Conda `LB2` environment and the same kernel as 03.

### 0.1 Imports and paths

**Implemented — Person 1.**

We prepared the libraries for data handling, plotting, RF selection, scaling, SVM classification and evaluation, following the imports in the SVM example. We kept relative paths from `notebooks/`, as in 03. We created the folders for this section's tables and figures.

The Kyte–Doolittle scale is imported in 1.2 using Biopython, listed in `requirements.txt`.

**Similar code:** imports in `04_SVM-sklearn_revised.ipynb`, section 0; relative paths and plot settings in 03 section 0.

In [1]:
# Imports follow section 0 in 04_SVM-sklearn_revised.ipynb.
# Path handling follows section 0 in 03_von_heijne_method.ipynb.
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    matthews_corrcoef,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
)

# These paths and plot settings follow section 0 in 03_von_heijne_method.ipynb.
input_dir = Path("../data/prepared")
table_dir = Path("../results/tables/04")
figure_dir = Path("../results/figures/04")

table_dir.mkdir(parents=True, exist_ok=True)
figure_dir.mkdir(parents=True, exist_ok=True)

sns.set_theme()


### 0.2 Load the prepared proteins

**Implemented — Person 1.**

We reused the FASTA reader and accession mapping from 03. The TSV files provide the metadata, and the matching FASTA files provide the sequences. We attached each sequence by accession and assigned label 1 to SP and 0 to NO_SP. The loaded tables contain 8,146 training proteins and 2,038 benchmarking proteins.

We kept the split and fold assignments from 02b. Cross-validation will use `data`, containing the training proteins; `benchmarking_data` will remain separate. Fold identifiers stay 1–5, with missing fold values allowed for benchmarking.

**Similar code:** 03 section 1.1 supplies this loader. The SVM example section 1 uses a random split of another dataset; we retained our prepared partitions instead.

In [2]:
# This reader is reused from section 1.1 in 03_von_heijne_method.ipynb.
def read_fasta(fasta_path):
    sequences = {}
    with open(fasta_path, encoding="utf-8") as fasta_in:
        for line in fasta_in:
            line = line.strip()
            if line.startswith(">"):
                accession = line[1:].split()[0]
                sequences[accession] = ""
            elif line:
                sequences[accession] += line
    return sequences


# TSV/FASTA loading and accession mapping also follow section 1.1 in that notebook.
positive_data = pd.read_csv(
    input_dir / "positive_dataset.tsv", sep="\t", dtype={"cv_fold": "Int64"}
)
negative_data = pd.read_csv(
    input_dir / "negative_dataset.tsv", sep="\t", dtype={"cv_fold": "Int64"}
)
positive_sequences = read_fasta(input_dir / "positive_dataset.fasta")
negative_sequences = read_fasta(input_dir / "negative_dataset.fasta")

positive_data["sequence"] = positive_data["accession"].map(positive_sequences)
negative_data["sequence"] = negative_data["accession"].map(negative_sequences)
positive_data["label"] = 1
negative_data["label"] = 0

# Keep the saved partitions, as in 03 section 1.1; no random split from the SVM example.
prepared_data = pd.concat([positive_data, negative_data], ignore_index=True)
data = prepared_data[prepared_data["split"] == "training"].copy()
benchmarking_data = prepared_data[prepared_data["split"] == "benchmarking"].copy()

print("Training proteins:", len(data))
print("Benchmarking proteins:", len(benchmarking_data))


Training proteins: 8146
Benchmarking proteins: 2038


## 1. Common feature definitions

We separated fixed choices from quantities learned in each training subset. The composition length n will be learned from positive training proteins; it is not a fixed 90-residue window (slides 30 and 36–40).

### 1.1 Estimate n from the current training subset

**Implemented — Person 1.**

We defined `get_n(train_data)` to estimate the composition length from the positive proteins in the supplied training subset (slides 30 and 40). Their `cleavage_pos` gives the last SP residue and therefore the annotated SP length.

We chose to round a half-integer median upward so n can be used for sequence slicing. This is our convention; the slides do not specify a rounding rule.

The function will be called separately for each three-fold inner-training set and again for the four-fold outer-training refit. The resulting n will also be used for that split's validation or test proteins. `cleavage_pos` contributes only to this training estimate, not to the predictor matrix.

**Similar code:** neither practical notebook includes this helper. Slide 40 specifies the calculation, and 02c establishes the meaning of `cleavage_pos`.

In [3]:
# This helper implements the training-only median in slides 30 and 40 of
# 04 - Methods - SVM.pdf; neither practical example contains an equivalent function.
def get_n(train_data):
    sp_lengths = train_data.loc[train_data["label"] == 1, "cleavage_pos"]
    median_length = sp_lengths.median()
    # Rounding upward is our project convention, not a rule from the examples.
    return int(np.ceil(median_length))


### 1.2 Fixed feature settings and residue policy

**Implemented — Person 1.**

We used the Kyte–Doolittle scale supplied by Biopython, as in the hydrophobicity example. Positive values indicate hydrophobic residues and negative values indicate hydrophilic residues. We chose the first 40 residues and windows of 5, following the starting example in slides 32–34. We also chose the first 5 residues for a charge-balance feature (2.3). For the regional features in 2.6, we chose a hydropathy threshold of 0, a downstream region of 5 positions and explicit small-residue sets. These are simple project choices, not fitted parameters or published optimal settings. For the propensity profiles in 2.4–2.5, we chose the first 40 residues with windows of 5 for alpha-helix propensity and 19 for transmembrane tendency. These window sizes are project choices. All these settings stay fixed during the k/C/gamma search; composition uses its separately estimated n.

We chose complete windows with uniform weights and no padding. The extractor preserves unknown positions and averages the known values within each original window. A window with no known values is missing, and the profile summaries use the non-missing values. Composition and charge balance exclude non-canonical residues from their denominators. Shorter sequences use the available N-terminal region.

We kept the canonical amino-acid alphabet in the same order as the composition columns. We kept the published scale dictionaries here with the common settings. Feature extraction belongs to section 2.

**Similar code:** setup and A.1–A.3 in `04_feature_extraction_hydrophobicity.ipynb` supply the scale and complete-window example. A.5 explains known-residue averaging; we adapted it to complete windows. The added regional settings are project choices, not implementations from that notebook.


In [4]:
# Scale lookup follows setup and A.1 in 04_feature_extraction_hydrophobicity.ipynb.
from Bio.SeqUtils.ProtParamData import kd

hydropathy_scale = kd
# Shared alphabet for our composition extractor; not supplied by the hydropathy example.
aa_alphabet = list("ACDEFGHIKLMNPQRSTVWY")

# Region and window follow the starting example on slide 34 of 04 - Methods - SVM.pdf.
hydropathy_length = 40
hydropathy_window = 5

# Fixed region for our charge proxy in 2.3; an extension of SVM slide 32.
charge_length = 5

# Fixed project settings for the regional extensions in 2.6; not class defaults.
hydropathy_threshold = 0.0
postpeak_length = 5
postpeak_small_residues = "AGS"
cleavage_small_residues = "ACGST"

# Additional scale profiles follow SVM slides 31 and 34; window sizes are our choices.
propensity_length = 40
alpha_helix_window = 5
tm_tendency_window = 19

# Published values from ProtScale; scale lookup follows A.1 in
# 04_feature_extraction_hydrophobicity.ipynb, replacing its KD dictionary.
# Chou-Fasman: https://web.expasy.org/protscale/pscale/alpha-helixFasman.html
alpha_helix_scale = {
    "A": 1.42, "C": 0.70, "D": 1.01, "E": 1.51, "F": 1.13,
    "G": 0.57, "H": 1.00, "I": 1.08, "K": 1.16, "L": 1.21,
    "M": 1.45, "N": 0.67, "P": 0.57, "Q": 1.11, "R": 0.98,
    "S": 0.77, "T": 0.83, "V": 1.06, "W": 1.08, "Y": 0.69,
}
# Zhao-London: https://web.expasy.org/protscale/pscale/Transmembranetendency.html
tm_tendency_scale = {
    "A": 0.38, "C": -0.30, "D": -3.27, "E": -2.90, "F": 1.98,
    "G": -0.19, "H": -1.44, "I": 1.97, "K": -3.46, "L": 1.82,
    "M": 1.40, "N": -1.62, "P": -1.44, "Q": -1.84, "R": -2.57,
    "S": -0.53, "T": -0.32, "V": 1.46, "W": 1.53, "Y": 0.49,
}



## 2. Feature extraction

We started with the 20 amino-acid fractions and two hydropathy summaries from slides 30–34. We then added charge balance, peak position, hydrophobic extent, post-peak small-residue fraction and cleavage-motif support; the user-run example produced 27 columns with no missing values.

Following slides 31 and 34, we added mean and maximum profiles for alpha-helix propensity and transmembrane tendency, giving 31 features. We executed the complete extractor and obtained 31 columns with no missing values; classifier evaluation is still pending. Sections 2.3–2.6 explain the extensions.

We ordered the groups as composition, hydropathy, charge and the two propensities proposed in the slides, followed by our additional sequence features. Specific formulas and scale choices are documented with their sources.


### 2.1 N-terminal amino-acid composition

**Implemented — Person 2.**

We defined `amino_acid_composition(sequence, n)`, which counts the canonical residues in the available first n positions and returns their fractions in the alphabet order `ACDEFGHIKLMNPQRSTVWY`. The 20 values are named `aa_fraction_A` to `aa_fraction_Y` and sum to 1 (slide 30).

Following the residue policy in 1.2, non-canonical residues are excluded from both the counts and the denominator, so each fraction describes the observed canonical residues rather than the full segment. Two training proteins (Q27972 and P02809) carry an X in this region; the remaining 8,144 are fully canonical, and no benchmarking protein is affected. A segment containing no canonical residue returns missing values rather than zeros, so an empty denominator cannot be reported as a composition of zeros.

The function receives n from its caller and never estimates it. It does not read `cleavage_pos`, so a protein's own annotation cannot choose its region; only `get_n` uses that column, on training positives.

We checked the function against the worked example on slide 30. For the first 21 residues of `MRFLAATFLLLALSTAAQAEPQF` it reproduces the published vector: A 0.29, L 0.24, F 0.10, T 0.10, and 0.05 for E, M, P, Q, R and S. For display, we listed all 20 amino acids in one table, including zero fractions and using two decimal places. The extractor's column order stays unchanged.

**Similar code:** the practical notebooks do not include this composition function. We adapted the canonical counting in 02c section 5.2 from pooled class counts to individual proteins; slide 30 defines the required vector.

In [5]:
composition_feature_names = [f"aa_fraction_{amino_acid}" for amino_acid in aa_alphabet]


# This function adapts the counting in section 5.2 of 02c_data_analysis.ipynb
# to individual proteins, producing the vector defined on slide 30 of the SVM slides.
def amino_acid_composition(sequence, n):
    """Fractions of the canonical residues within the first n positions."""
    counts = dict.fromkeys(aa_alphabet, 0)
    observed = 0
    for residue in sequence[:n]:
        if residue in counts:
            counts[residue] += 1
            observed += 1
    if observed == 0:
        return {name: np.nan for name in composition_feature_names}
    return {
        f"aa_fraction_{amino_acid}": counts[amino_acid] / observed
        for amino_acid in aa_alphabet
    }


# This sequence and n=21 reproduce the worked example on slide 30 of 04 - Methods - SVM.pdf.
slide_example = amino_acid_composition("MRFLAATFLLLALSTAAQAEPQF", 21)
composition_table = pd.DataFrame(
    [[slide_example[f"aa_fraction_{amino_acid}"] for amino_acid in aa_alphabet]],
    columns=aa_alphabet,
    index=["Fraction"],
)

print("N-terminal amino acid composition for slide example:")

with pd.option_context("display.max_columns", 20):
    display(composition_table.round(2))

print("Fractions sum to:", round(sum(slide_example.values()), 12))


N-terminal amino acid composition for slide example:


,A,C,D,E,F,G,H,I,K,L,M,N,P,Q,R,S,T,V,W,Y
Fraction,0.29,0.0,0.0,0.05,0.1,0.0,0.0,0.0,0.0,0.24,0.05,0.0,0.05,0.05,0.05,0.05,0.1,0.0,0.0,0.0


Fractions sum to: 1.0


### 2.2 Hydropathy profile and summaries

**Implemented — Person 2.**

We wrote the profile explicitly instead of calling `protein_scale`, because Biopython handles residues outside the scale by warning and skipping their symmetric partners within a window, whereas our policy in 1.2 averages the known values without shifting positions. `hydropathy_profile(sequence)` takes the first 40 residues, maps each to its Kyte–Doolittle value, and averages the known values inside every complete 5-position window with uniform weights and no padding. A canonical segment of length L therefore gives L − w + 1 window values; the shortest training protein is exactly 40 residues long, so every training profile contains 36 values. A window whose five positions are all unknown is recorded as missing.

`hydropathy_features(sequence)` summarizes the non-missing window values as `hydropathy_max_w5` and `hydropathy_profile_mean_w5` (slide 34). The mean is taken over the window profile rather than over the 40 residue values directly, so the first and last four residues contribute to fewer windows than the interior ones.

We compared the first 300 fully canonical N-termini with `ProteinAnalysis.protein_scale(kd, 5)` and obtained a largest difference of 8.9e-16. The output also shows missing values for both features on 40 unknown residues. We reused the window calculation in `scale_profile(sequence, scale, window, length)` and kept `hydropathy_profile` as the KD wrapper. Replacing a residue with X would preserve the profile length and could affect only windows containing that position; this follows from keeping the original positions when forming windows.

The maximum and the mean describe the same profile from two directions. The maximum reports the strongest local hydrophobic peak, which may reflect an SP h-region or another hydrophobic segment, and the mean reports how hydrophobic the N-terminus is overall; a short strong peak and a broad mild one can share a mean but not a maximum.

**Similar code:** `window_profile` in hydrophobicity notebook A.2 is the closest loop, and A.3 compares it with `ProteinAnalysis.protein_scale`. A.5 explains the unknown-residue issue, which we handled with the complete windows chosen in 1.2 rather than its shrinking windows; B.4 shows the profile-to-feature aggregation. Slide 34 gives the max/mean pair.

In [6]:
hydropathy_feature_names = [
    f"hydropathy_max_w{hydropathy_window}",
    f"hydropathy_profile_mean_w{hydropathy_window}",
]


# This function combines A.1 (residue values) and A.2 (window_profile) in
# 04_feature_extraction_hydrophobicity.ipynb, with the unknown-residue policy from A.5.
def scale_profile(sequence, scale, window, length):
    """Complete-window means for a residue scale, retaining unknown positions."""
    # Similar to A.1: look up one scale value per residue, retaining unknown positions.
    values = [
        scale.get(residue, np.nan)
        for residue in sequence[:length]
    ]
    profile = []
    # Similar to window_profile in A.2: slide only complete windows along the sequence.
    for start in range(len(values) - window + 1):
        # Adapted from A.5: average known residues, without shrinking the end windows.
        window_values = [
            value
            for value in values[start : start + window]
            if not np.isnan(value)
        ]
        profile.append(float(np.mean(window_values)) if window_values else np.nan)
    return profile


# KD wrapper around the shared A.2-style loop; existing feature functions keep
# the same interface and fixed settings from SVM slide 34.
def hydropathy_profile(sequence):
    """Complete-window KD profile over the fixed N-terminal region."""
    return scale_profile(sequence, hydropathy_scale, hydropathy_window, hydropathy_length)


# This function follows profile aggregation in B.4 of
# 04_feature_extraction_hydrophobicity.ipynb and the max/mean pair on SVM slide 34.
def hydropathy_features(sequence):
    """Maximum and mean of the non-missing window values of the profile."""
    profile = [value for value in hydropathy_profile(sequence) if not np.isnan(value)]
    if not profile:
        return {name: np.nan for name in hydropathy_feature_names}
    return dict(zip(hydropathy_feature_names, [max(profile), float(np.mean(profile))]))


# This comparison follows A.3 in 04_feature_extraction_hydrophobicity.ipynb:
# compare our explicit loop with ProteinAnalysis.protein_scale on canonical sequences.
from Bio.SeqUtils.ProtParam import ProteinAnalysis

canonical_sequences = [
    sequence
    for sequence in data["sequence"]
    if set(sequence[:hydropathy_length]) <= set(aa_alphabet)
]
largest_difference = max(
    abs(ours - reference)
    for sequence in canonical_sequences[:300]
    for ours, reference in zip(
        hydropathy_profile(sequence),
        ProteinAnalysis(sequence[:hydropathy_length]).protein_scale(
            hydropathy_scale, hydropathy_window
        ),
    )
)

print(f"Canonical N-termini: {len(canonical_sequences)} of {len(data)} training proteins")
print(f"Largest difference from protein_scale (first 300): {largest_difference:.1e}")
print("Profile values per 40-residue segment:", len(hydropathy_profile("A" * 40)))
# Project check for the missing-value behavior motivated by A.5.
print("All-unknown segment:", hydropathy_features("X" * 40))

Canonical N-termini: 8144 of 8146 training proteins
Largest difference from protein_scale (first 300): 8.9e-16
Profile values per 40-residue segment: 36
All-unknown segment: {'hydropathy_max_w5': nan, 'hydropathy_profile_mean_w5': nan}


### 2.3 N-terminal charge

**Implemented — Person 2.**

Charge is one of the feature categories proposed on slide 31. We chose `nterminal_charge(sequence)` to calculate `(K + R - D - E counts) / canonical residue count` over the first 5 positions. Other canonical residues, including H, score zero; unknown residues are excluded from the denominator. `charge_features(sequence)` returns this value in a named dictionary for the assembled extractor.

[Guo et al. (2018)](https://pubmed.ncbi.nlm.nih.gov/29229776/) found a role for positive n-region charge in the post-translational translocation of small secretory preproteins. Our formula and first-5 region are project choices, not a universal SP rule or a reproduction of that experiment. The result is a charge proxy, not physical net charge at a specified pH; its predictive value still needs evaluation.

**Similar code:** charge counting resembles the composition loop in 2.1. Neither class notebook supplies this exact formula; slide 31 proposes charge and slide 32 illustrates the positive n-region.


In [7]:
charge_feature_names = [f"nterminal_charge_first{charge_length}"]


# Project extension: residue counting like 2.1, motivated by SVM slide 32 and
# Guo et al. (2018); no equivalent charge function in the class notebooks.
def nterminal_charge(sequence):
    """Mean K/R (+1) and D/E (-1) proxy over canonical residues in the first 5 positions."""
    residues = [residue for residue in sequence[:charge_length] if residue in aa_alphabet]
    if not residues:
        return np.nan
    positive = sum(residue in "KR" for residue in residues)
    negative = sum(residue in "DE" for residue in residues)
    return (positive - negative) / len(residues)


# Project grouping, like the feature dictionary in B.4 of
# 04_feature_extraction_hydrophobicity.ipynb; charge is proposed on SVM slide 31.
def charge_features(sequence):
    """N-terminal charge proxy as a named feature dictionary."""
    return {charge_feature_names[0]: nterminal_charge(sequence)}


### 2.4 Alpha-helix propensity profile and summaries

**Implemented — Person 2.**

We chose the [Chou–Fasman alpha-helix scale from ProtScale](https://web.expasy.org/protscale/pscale/alpha-helixFasman.html), based on amino-acid frequencies in known structures. Its values were included in the shared definitions in 1.2. Higher values indicate greater alpha-helix propensity.

`alpha_helix_profile(sequence)` uses the first 40 residues and complete windows of 5, with the same unknown-residue policy as hydropathy. `alpha_helix_features(sequence)` returns the mean and maximum of the non-missing window values. An empty or entirely missing profile gives NaN for both features. These summaries describe sequence propensity; they do not assign secondary structure to individual residues or implement the full Chou–Fasman prediction method.

We chose the region and window size before evaluation and kept them fixed. Their value for SP classification still needs CV evaluation.

**Similar code:** A.1–A.2 in `04_feature_extraction_hydrophobicity.ipynb` map scale values and build a window profile; B.4 summarizes it. Slide 34 applies this approach to other scales. We followed the same profile-and-summary structure as 2.2, using `scale_profile` with the alpha-helix settings.


In [8]:
alpha_helix_feature_names = [
    f"alpha_helix_profile_mean_w{alpha_helix_window}",
    f"alpha_helix_max_w{alpha_helix_window}",
]


# Same A.2-style profile as 2.2, with another scale as proposed on SVM slide 34.
# The window loop comes from 04_feature_extraction_hydrophobicity.ipynb.
def alpha_helix_profile(sequence):
    """Complete-window alpha-helix propensity profile over the fixed N-terminus."""
    return scale_profile(sequence, alpha_helix_scale, alpha_helix_window, propensity_length)


# Same profile aggregation as 2.2 and B.4 of
# 04_feature_extraction_hydrophobicity.ipynb.
def alpha_helix_features(sequence):
    """Mean and maximum of the non-missing alpha-helix window values."""
    profile = [value for value in alpha_helix_profile(sequence) if not np.isnan(value)]
    if not profile:
        return {name: np.nan for name in alpha_helix_feature_names}
    return dict(zip(alpha_helix_feature_names, [float(np.mean(profile)), max(profile)]))


### 2.5 Transmembrane tendency profile and summaries

**Implemented — Person 2.**

We chose the [Zhao–London transmembrane-tendency scale from ProtScale](https://web.expasy.org/protscale/pscale/Transmembranetendency.html). Their [2006 study](https://doi.org/10.1110/ps.062286306) derived it by comparing transmembrane and soluble sequences. We included its values in 1.2; higher values indicate greater transmembrane tendency.

`tm_tendency_profile(sequence)` uses the first 40 residues and complete windows of 19. We chose the longer window to describe a broader segment and kept it fixed before evaluation. `tm_tendency_features(sequence)` returns the mean and maximum of the non-missing window values, with NaN for both when the profile is empty or entirely missing. These are propensity indices, not probabilities or predictions of membrane topology. Their value for SP classification still needs CV evaluation.

**Similar code:** A.1–A.2 and B.4 in `04_feature_extraction_hydrophobicity.ipynb` provide the scale-profile-summary approach. We followed the same structure as 2.2 and 2.4, changing the scale and window as proposed on SVM slide 34.


In [9]:
tm_tendency_feature_names = [
    f"tm_tendency_profile_mean_w{tm_tendency_window}",
    f"tm_tendency_max_w{tm_tendency_window}",
]


# Same A.2-style profile as 2.2, with another scale as proposed on SVM slide 34.
# The window loop comes from 04_feature_extraction_hydrophobicity.ipynb.
def tm_tendency_profile(sequence):
    """Complete-window transmembrane-tendency profile over the fixed N-terminus."""
    return scale_profile(sequence, tm_tendency_scale, tm_tendency_window, propensity_length)


# Same profile aggregation as 2.2 and B.4 of
# 04_feature_extraction_hydrophobicity.ipynb.
def tm_tendency_features(sequence):
    """Mean and maximum of the non-missing transmembrane window values."""
    profile = [value for value in tm_tendency_profile(sequence) if not np.isnan(value)]
    if not profile:
        return {name: np.nan for name in tm_tendency_feature_names}
    return dict(zip(tm_tendency_feature_names, [float(np.mean(profile)), max(profile)]))


### 2.6 Additional sequence features

**Implemented — Person 2.**

After the feature categories proposed in the slides, we grouped four project extensions inspired by the literature. The categories in 2.3–2.5 also use published sources for our specific implementations.

`hydropathy_peak_position(sequence)` returns the 1-based center of the first maximum window in the KD profile. Missing windows retain their positions, and an empty or entirely missing profile gives NaN. [Imai et al. (2011)](https://link.springer.com/article/10.1186/1471-2164-12-79) used a hydropathy peak to align SP-related regions in a mitochondrial-protein classifier; our use of its position as a feature is a simpler project adaptation, not a cleavage-site prediction.

We also added three regional summaries. [Käll et al. (2004)](https://sonnhammer.org/download/papers/2004_JMB_338%3A1027-1036.pdf) model SP and transmembrane regions together in Phobius, motivating information beyond peak height. `hydrophobic_stretch_length` finds the longest run of profile windows with a KD mean greater than 0. A run of r overlapping windows spans r + w − 1 residue positions; no qualifying window gives 0. Missing windows break a run, and an empty or entirely missing profile gives NaN. The threshold of 0 is our fixed sign-based convention, not a published optimum or a boundary detector.

`postpeak_small_fraction` counts A/G/S among the canonical residues in the next 5 positions after the end of the maximum window, stopping at position 40. This is inspired by the c-region descriptor in [Imai et al. (2011)](https://link.springer.com/article/10.1186/1471-2164-12-79), with our own region definition. No downstream positions gives 0; an available region containing only unknowns gives NaN. The end of the maximum window is only an anchor, not the inferred end of an h-region.

`cleavage_motif_score` scans possible bonds from the end of the maximum window through position 40, requiring a following residue in the protein. For a bond after residue s (1-based), it scores residues s − 2 and s, corresponding to −3 and −1. Each A/C/G/S/T contributes 0.5; the largest score is retained. Sites with an unknown residue at either scored position are skipped, and no scorable bond gives 0. A missing peak gives NaN.

Small, uncharged residues at −3 and −1 motivate this proxy ([von Heijne, 1984](https://pubmed.ncbi.nlm.nih.gov/6423828/)). We chose the five-residue set and equal weights as a coarse approximation. Scores of 0, 0.5 and 1 describe observed motif support; they are not probabilities or cleavage predictions, and chance matches are possible. All four descriptors use fixed rules and sequence input; no annotated cleavage sites, learned matrices or extra search parameters are used.

`additional_features(sequence)` collects these four descriptors in a dictionary, so they enter the assembled extractor in the same way as the other feature groups.

**Similar code:** `hydrophobic_stretch_length` extends the profile aggregation in B.4 of `04_feature_extraction_hydrophobicity.ipynb`. The A/G/S fraction uses counting like 2.1. Motif scanning uses a sequence loop like `score_sequence` in section 3 of `03_von_heijne_method.ipynb`, but no fitted PSWM or scoring function was copied. Neither class notebook provides these four exact formulas.


In [10]:
additional_feature_names = [
    "hydropathy_peak_position",
    "hydrophobic_stretch_length",
    "postpeak_small_fraction",
    "cleavage_motif_score",
]


# Project extension of the profile summaries in B.4 of
# 04_feature_extraction_hydrophobicity.ipynb; 2.2 supplies the A.2-style profile.
def hydropathy_peak_position(sequence):
    """1-based center of the first maximum window, preserving unknown positions."""
    profile = hydropathy_profile(sequence)
    if not profile or np.all(np.isnan(profile)):
        return np.nan
    peak_start = int(np.nanargmax(profile))
    return peak_start + hydropathy_window // 2 + 1


# Project extension of B.4 in 04_feature_extraction_hydrophobicity.ipynb:
# summarize contiguous windows rather than just their maximum or mean.
def hydrophobic_stretch_length(sequence):
    """Residue span covered by the longest run of above-threshold KD windows."""
    profile = hydropathy_profile(sequence)
    if not profile or np.all(np.isnan(profile)):
        return np.nan
    current_run = 0
    longest_run = 0
    for value in profile:
        if value > hydropathy_threshold:
            current_run += 1
            longest_run = max(longest_run, current_run)
        else:
            current_run = 0
    return longest_run + hydropathy_window - 1 if longest_run else 0


# Project counting extension, like composition in 2.1; regional idea from
# Imai et al. (2011), not an exact function from the class notebooks.
def postpeak_small_fraction(sequence):
    """A/G/S fraction just after the maximum window, within the fixed N-terminus."""
    peak = hydropathy_peak_position(sequence)
    if np.isnan(peak):
        return np.nan
    window_end = int(peak) + hydropathy_window // 2
    region = sequence[window_end : min(window_end + postpeak_length, hydropathy_length)]
    if not region:
        return 0.0
    residues = [residue for residue in region if residue in aa_alphabet]
    if not residues:
        return np.nan
    return sum(residue in postpeak_small_residues for residue in residues) / len(residues)


# Project motif scan: like score_sequence in section 3 of 03_von_heijne_method.ipynb,
# but only fixed -3/-1 membership scores, without its trained PSWM.
def cleavage_motif_score(sequence):
    """Best small-residue support at -3/-1 among sequence-derived candidate bonds."""
    peak = hydropathy_peak_position(sequence)
    if np.isnan(peak):
        return np.nan
    first_site = max(int(peak) + hydropathy_window // 2, 3)
    last_site = min(hydropathy_length, len(sequence) - 1)
    scores = []
    for site in range(first_site, last_site + 1):
        residues = [sequence[site - 3], sequence[site - 1]]
        if all(residue in aa_alphabet for residue in residues):
            scores.append(sum(residue in cleavage_small_residues for residue in residues) / 2)
    return max(scores) if scores else 0.0


# Project feature dictionary, like B.4 of
# 04_feature_extraction_hydrophobicity.ipynb, collecting our four extensions.
def additional_features(sequence):
    """Peak position, hydrophobic extent and downstream context features."""
    return {
        additional_feature_names[0]: hydropathy_peak_position(sequence),
        additional_feature_names[1]: hydrophobic_stretch_length(sequence),
        additional_feature_names[2]: postpeak_small_fraction(sequence),
        additional_feature_names[3]: cleavage_motif_score(sequence),
    }


### 2.7 Assemble the feature extractor and catalog

**Implemented — Person 2.**

`extract_features(proteins, n)` returns an accession-indexed DataFrame with 20 composition columns, two hydropathy summaries, charge, the alpha-helix and transmembrane summaries, then the four descriptors in `additional_feature_names`. We kept this order in `feature_names`, giving `D = 31` for the k grid (slide 38). Each group returns a dictionary, which we add to the protein's record with `record.update(...)`. Row order follows the supplied table, keeping features aligned with its labels.

Only predictors calculated from the sequence leave this function. Labels, folds, organism, `cleavage_pos` and `has_tm_in_90` stay in the metadata. Composition depends on the supplied n; the other eleven features use fixed regions and do not depend on the annotated SP length.

The catalog records the definition, region, scale, window, aggregation and unit of each feature for export in 6.2. The example below produced an 8,146 × 31 matrix with no missing values. It uses the full training split only to demonstrate extraction, without fitting a classifier. During CV, sections 4 and 5 will estimate n from each training subset and extract the corresponding features; they will not reuse `example_features`.

**Similar code:** B.4 in `04_feature_extraction_hydrophobicity.ipynb` assembles a feature dictionary, but does not build a protein matrix. This wrapper is our adaptation. SVM slide 36 combines features, and slide 38 ties D to the k grid.


In [11]:
# Project feature order; D is the feature count used for the grid on SVM slide 38.
feature_names = (
    composition_feature_names + hydropathy_feature_names + charge_feature_names
    + alpha_helix_feature_names + tm_tendency_feature_names + additional_feature_names
)
D = len(feature_names)


# Project wrapper around the composition and profile extractors; no direct equivalent
# in the examples. B.4 motivates fixed-size features, and SVM slide 36 combines them.
def extract_features(proteins, n):
    """Sequence features in the shared column order, indexed by accession."""
    records = []
    for sequence in proteins["sequence"]:
        record = {}
        record.update(amino_acid_composition(sequence, n))
        record.update(hydropathy_features(sequence))
        record.update(charge_features(sequence))
        record.update(alpha_helix_features(sequence))
        record.update(tm_tendency_features(sequence))
        record.update(additional_features(sequence))
        records.append(record)
    features = pd.DataFrame(
        records, index=pd.Index(proteins["accession"], name="accession")
    )
    return features[feature_names]


# Project documentation table for our feature definitions:
feature_catalog = pd.DataFrame(
    [
        {
            "feature": f"aa_fraction_{amino_acid}",
            "definition": f"Fraction of the canonical residues that are {amino_acid}",
            "region": "first n residues (n estimated from the training positives)",
            "scale": "none (residue counts)",
            "window": pd.NA,
            "aggregation": "fraction",
            "unit": "fraction of canonical residues",
        }
        for amino_acid in aa_alphabet
    ]
    + [
        {
            "feature": hydropathy_feature_names[0],
            "definition": "Maximum of the Kyte-Doolittle window profile",
            "region": f"first {hydropathy_length} residues (fixed)",
            "scale": "Kyte-Doolittle",
            "window": hydropathy_window,
            "aggregation": "maximum",
            "unit": "hydropathy index",
        },
        {
            "feature": hydropathy_feature_names[1],
            "definition": "Mean of the Kyte-Doolittle window profile",
            "region": f"first {hydropathy_length} residues (fixed)",
            "scale": "Kyte-Doolittle",
            "window": hydropathy_window,
            "aggregation": "mean",
            "unit": "hydropathy index",
        },
        {
            "feature": charge_feature_names[0],
            "definition": "(K + R - D - E counts) / canonical residue count; H scores zero",
            "region": f"first {charge_length} residues (fixed)",
            "scale": "K/R: +1; D/E: -1; other canonical residues: 0",
            "window": pd.NA,
            "aggregation": "mean charge proxy",
            "unit": "dimensionless (not physical net charge)",
        },
        {
            "feature": alpha_helix_feature_names[0],
            "definition": "Mean of the Chou-Fasman alpha-helix window profile",
            "region": f"first {propensity_length} residues (fixed)",
            "scale": "Chou-Fasman alpha-helix",
            "window": alpha_helix_window,
            "aggregation": "mean",
            "unit": "propensity index (not a probability)",
        },
        {
            "feature": alpha_helix_feature_names[1],
            "definition": "Maximum of the Chou-Fasman alpha-helix window profile",
            "region": f"first {propensity_length} residues (fixed)",
            "scale": "Chou-Fasman alpha-helix",
            "window": alpha_helix_window,
            "aggregation": "maximum",
            "unit": "propensity index (not a probability)",
        },
        {
            "feature": tm_tendency_feature_names[0],
            "definition": "Mean of the Zhao-London transmembrane tendency window profile",
            "region": f"first {propensity_length} residues (fixed)",
            "scale": "Zhao-London transmembrane tendency",
            "window": tm_tendency_window,
            "aggregation": "mean",
            "unit": "propensity index (not a probability)",
        },
        {
            "feature": tm_tendency_feature_names[1],
            "definition": "Maximum of the Zhao-London transmembrane tendency window profile",
            "region": f"first {propensity_length} residues (fixed)",
            "scale": "Zhao-London transmembrane tendency",
            "window": tm_tendency_window,
            "aggregation": "maximum",
            "unit": "propensity index (not a probability)",
        },
        {
            "feature": additional_feature_names[0],
            "definition": "1-based center of the first maximum in the KD window profile",
            "region": f"first {hydropathy_length} residues (fixed)",
            "scale": "Kyte-Doolittle",
            "window": hydropathy_window,
            "aggregation": "position of maximum (first tie)",
            "unit": "residue position (1-based window center)",
        },
        {
            "feature": additional_feature_names[1],
            "definition": "Longest run of KD windows above threshold, converted to residue span",
            "region": f"first {hydropathy_length} residues (fixed)",
            "scale": f"Kyte-Doolittle > {hydropathy_threshold}",
            "window": hydropathy_window,
            "aggregation": "longest window run + window size - 1; 0 if no run",
            "unit": "residue span (not an annotated h-region length)",
        },
        {
            "feature": additional_feature_names[2],
            "definition": f"Fraction of canonical residues in {postpeak_small_residues}",
            "region": f"up to {postpeak_length} positions after the maximum window, within first {hydropathy_length}",
            "scale": f"membership in {postpeak_small_residues}",
            "window": pd.NA,
            "aggregation": "fraction; 0 if no downstream positions",
            "unit": "fraction of canonical residues",
        },
        {
            "feature": additional_feature_names[3],
            "definition": f"Best fraction of -3/-1 residues in {cleavage_small_residues}",
            "region": f"candidate bonds from the maximum window end through position {hydropathy_length}, with a following residue",
            "scale": f"membership in {cleavage_small_residues}",
            "window": pd.NA,
            "aggregation": "maximum of 0, 0.5 or 1; 0 if no scorable bond",
            "unit": "motif compatibility (not a probability)",
        },
    ]
)

# Example extraction: a project demonstration of the assembled matrix.
# CV will estimate n separately in each training subset, as on SVM slide 40.
example_features = extract_features(data, get_n(data))

print("Example extraction:", example_features.shape)
print("Missing values:", int(example_features.isna().sum().sum()))

feature_catalog


Example extraction: (8146, 31)
Missing values: 0


,feature,definition,region,scale,window,aggregation,unit
0,aa_fraction_A,Fraction of the canonical residues that are A,first n residues (n estimated from the trainin...,none (residue counts),<NA>,fraction,fraction of canonical residues
1,aa_fraction_C,Fraction of the canonical residues that are C,first n residues (n estimated from the trainin...,none (residue counts),<NA>,fraction,fraction of canonical residues
2,aa_fraction_D,Fraction of the canonical residues that are D,first n residues (n estimated from the trainin...,none (residue counts),<NA>,fraction,fraction of canonical residues
3,aa_fraction_E,Fraction of the canonical residues that are E,first n residues (n estimated from the trainin...,none (residue counts),<NA>,fraction,fraction of canonical residues
4,aa_fraction_F,Fraction of the canonical residues that are F,first n residues (n estimated from the trainin...,none (residue counts),<NA>,fraction,fraction of canonical residues
5,aa_fraction_G,Fraction of the canonical residues that are G,first n residues (n estimated from the trainin...,none (residue counts),<NA>,fraction,fraction of canonical residues
6,aa_fraction_H,Fraction of the canonical residues that are H,first n residues (n estimated from the trainin...,none (residue counts),<NA>,fraction,fraction of canonical residues
7,aa_fraction_I,Fraction of the canonical residues that are I,first n residues (n estimated from the trainin...,none (residue counts),<NA>,fraction,fraction of canonical residues
8,aa_fraction_K,Fraction of the canonical residues that are K,first n residues (n estimated from the trainin...,none (residue counts),<NA>,fraction,fraction of canonical residues
9,aa_fraction_L,Fraction of the canonical residues that are L,first n residues (n estimated from the trainin...,none (residue counts),<NA>,fraction,fraction of canonical residues


## 3. Random Forest selection and SVM helpers

We followed the order in slides 35–36: extract features, rank them with Random Forest, keep top-k, standardize selected columns, and train an RBF SVM. Person 3 prepares the helpers; Person 4 will call them inside CV.

### 3.1 Fit a Random Forest ranking

**Implemented — Person 3.**

We defined `rank_features(X_train, y_train)` following slide 35. The function fits a Random Forest with 500 trees and `random_state = 0`, then sorts the features from highest to lowest importance. It returns their column indices, names and importances in that order.

Person 4 will call it once per current training subset and reuse the ranking for all candidate k, C and gamma values. The same ranking can provide the top 8, top 16 or all 31 features without fitting another forest.

**Similar code:** slide 35 in `04 - Methods - SVM.pdf` provides the RF fit and importance ranking. We kept that approach and added the ordered names and importances for later reporting.

In [12]:
# This function follows RF fitting and importance ranking on slide 35 of
# 04 - Methods - SVM.pdf; names and sorted importances are added for reporting.
def rank_features(X_train, y_train):
    rf = RandomForestClassifier(n_estimators=500, random_state=0)
    rf.fit(X_train, y_train)

    importances = rf.feature_importances_
    ranking = np.argsort(importances)[::-1]
    ranked_features = X_train.columns[ranking].tolist()
    ranked_importances = importances[ranking]

    return ranking, ranked_features, ranked_importances


### 3.2 Build the coupled parameter grid

**Implemented — Person 3.**

We followed the minimal grid on slide 38. The candidate feature counts are the ceilings of D/4 and D/2, plus all D features. With 31 features, this gives k = 8, 16 and 31. We combined these with C = 1, 4 and 16, and gamma = 0.5/k, 1/k and 2/k.

The loops build 27 `(k, C, gamma)` tuples, considering smaller k first. Person 4 will compare their mean inner-validation MCC and retain the first tuple if scores tie exactly. The grid can be extended later using inner-validation results, as described in 4.3.

**Similar code:** slide 38 in `04 - Methods - SVM.pdf` defines the grid, and slide 40 shows the candidate loop. Sections 4.2 and 6 of `04_SVM-sklearn_revised.ipynb` explain gamma but do not build this search.

In [13]:
# These candidates follow the minimal grid on slide 38 of 04 - Methods - SVM.pdf.
k_values = sorted({int(np.ceil(D / 4)), int(np.ceil(D / 2)), D})
C_values = [1, 4, 16]
gamma_factors = [0.5, 1, 2]

parameter_grid = []
for k in k_values:
    for C in C_values:
        for factor in gamma_factors:
            gamma = factor / k
            parameter_grid.append((k, C, gamma))

print("Feature counts:", k_values)
print("Regularization constants:", C_values)
print("Gamma factors:", gamma_factors)
print("Parameter combinations:", len(parameter_grid))

for k, C, gamma in parameter_grid:
    print(f"Parameters: k={k}, C={C}, gamma={gamma:.4f}")



Feature counts: [8, 16, 31]
Regularization constants: [1, 4, 16]
Gamma factors: [0.5, 1, 2]
Parameter combinations: 27
Parameters: k=8, C=1, gamma=0.0625
Parameters: k=8, C=1, gamma=0.1250
Parameters: k=8, C=1, gamma=0.2500
Parameters: k=8, C=4, gamma=0.0625
Parameters: k=8, C=4, gamma=0.1250
Parameters: k=8, C=4, gamma=0.2500
Parameters: k=8, C=16, gamma=0.0625
Parameters: k=8, C=16, gamma=0.1250
Parameters: k=8, C=16, gamma=0.2500
Parameters: k=16, C=1, gamma=0.0312
Parameters: k=16, C=1, gamma=0.0625
Parameters: k=16, C=1, gamma=0.1250
Parameters: k=16, C=4, gamma=0.0312
Parameters: k=16, C=4, gamma=0.0625
Parameters: k=16, C=4, gamma=0.1250
Parameters: k=16, C=16, gamma=0.0312
Parameters: k=16, C=16, gamma=0.0625
Parameters: k=16, C=16, gamma=0.1250
Parameters: k=31, C=1, gamma=0.0161
Parameters: k=31, C=1, gamma=0.0323
Parameters: k=31, C=1, gamma=0.0645
Parameters: k=31, C=4, gamma=0.0161
Parameters: k=31, C=4, gamma=0.0323
Parameters: k=31, C=4, gamma=0.0645
Parameters: k=31, C=

### 3.3 Standardize selected features and fit SVC

**Implemented — Person 3.**

We defined `fit_svm(X_train, y_train, C, gamma)` following section 3 of the SVM example. It fits a scaler on the supplied training columns, transforms them and trains an RBF SVM with the candidate C and gamma. It returns the fitted scaler and model together.

`predict_svm(X, scaler, model)` applies that same scaler before predicting validation or test labels. Person 4 will pass the top-k columns selected from the current training ranking, keeping the same column order for training and prediction.

**Similar code:** section 3 of `04_SVM-sklearn_revised.ipynb` supplies the scaler, training transformation, SVC fit and test transformation. We wrapped those steps in two functions for reuse in the loops on slide 40 of `04 - Methods - SVM.pdf`.

In [14]:
# These functions wrap the scaling, fitting and prediction steps in section 3
# of 04_SVM-sklearn_revised.ipynb for reuse inside the CV loops.
def fit_svm(X_train, y_train, C, gamma):
    scaler = StandardScaler().fit(X_train)
    model = SVC(C=C, kernel="rbf", gamma=gamma)
    model.fit(scaler.transform(X_train), y_train)
    return scaler, model


def predict_svm(X, scaler, model):
    return model.predict(scaler.transform(X))


### 3.4 Calculate classification metrics

**Implemented — Person 3.**

We adapted the example's `evaluate` function to return a dictionary with precision, recall, accuracy, MCC and F1, using the same metric names as 03. Class 1 represents signal peptides. We chose zero for undefined precision, recall or F1, for example when a model predicts no positives.

Person 4 will use MCC to compare inner-validation candidates and report all five metrics for each outer test fold. The helpers in section 3 are now prepared for those loops; no model has been fitted by defining them.

**Similar code:** `evaluate` in section 1 of `04_SVM-sklearn_revised.ipynb` calculates the same five metrics. We return the values instead of printing them so the CV loops can save them in tables, as in 03 section 5.

In [15]:
# This function adapts evaluate from section 1 of 04_SVM-sklearn_revised.ipynb:
# return a record for CV tables rather than printing each metric.
def calculate_metrics(y_true, y_pred):
    return {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }


## 4. Inner model selection

We chose explicit loops from slide 40 so the changing composition length and reused RF ranking are visible. Selection will use the four saved folds available within each outer-training set, with three for training and one for validation.

### 4.1 Select outer and inner subsets

**Pending — Person 4; receives the completed helpers from Person 3.** Define the five outer rounds from saved `cv_fold` identifiers 1–5. For each round, hold one fold out as test and use the other four as outer training. Within those four, choose each in turn as inner validation and use the other three for inner training.

**Deliverable:** subset selection for the nested loops. The slides label subsets 0–4; our existing 1–5 labels identify the same five-fold structure and remain unchanged. Benchmarking is excluded.

**Similar code:** slides 37 and 40 describe these subsets. Neither practical notebook implements the full nested protocol. The cyclic `get_cv_split` from 03 needs replacement because it uses one validation fold per outer round.

### 4.2 Prepare each inner-training fit

**Pending — Person 4.** For each inner split, compute n from its three-fold training positives using `get_n`. Call `extract_features` on inner training and validation with that same n. Fit one RF ranking using inner-training features and labels.

**Deliverable:** training/validation matrices, aligned labels, n and the ranking for the current inner split. These can be retained within the current outer round while scoring the grid. The next inner split must estimate its own n and ranking.

**Similar code:** slide 40 gives this exact sequence. The feature functions from the hydrophobicity notebook support extraction, but the training-dependent composition length is our adaptation to the practical instructions.

### 4.3 Score the grid and choose a combination

**Pending — Person 4.** For each `(k, C, gamma)` candidate, take the first k ranked columns, fit the scaler/SVM on inner training, predict inner validation and record MCC in `inner_scores`. Average each candidate's MCC across the four validation folds and choose the highest mean, using the tie convention in 3.2.

Before the outer refit, inspect whether the optimum lies on a searchable grid boundary. Extend the relevant candidate range using only inner-validation results, as required by slide 38, and record the final grid used. k cannot exceed D; selecting all D columns is a dimensional limit. Reuse the appropriate inner-training matrices and rankings when adding candidates.

**Deliverable:** best k/C/gamma for the current outer round and the candidate scores. Inner maxima are selection scores, not performance estimates. All choices must be complete before predicting the outer test fold.

**Similar code:** slides 38–40 are the primary reference. The practical SVM notebook does not include this search; it deliberately fixes parameters to explain SVC behavior.

## 5. Outer refit and evaluation

We planned one external test per saved fold, after all inner choices have been made. The procedure will be repeated five times, without repartitioning the data (slides 37–41).

### 5.1 Refit on the four outer-training folds

**Pending — Person 4.** Recompute n using all positive proteins in the current four-fold outer-training set. Extract outer-training and outer-test features with that n. Fit a new RF ranking on outer training, select its best k columns, and refit the scaler/SVM with the chosen C and gamma.

**Deliverable:** the final fitted procedure for this round, its n, selected feature names and their ranks/importances. Do not reuse one inner split's n, ranking or model as the final refit.

**Similar code:** the refit block in slide 40 specifies every step. SVM notebook section 3 supports scaler/SVC fitting, and hydrophobicity notebook A/B.4 supports profile extraction and aggregation.

### 5.2 Predict and record all five external folds

**Pending — Person 4.** Predict the held-out outer test proteins once with the refitted procedure and calculate the five test metrics. Repeat sections 4–5 for all saved test folds. Preserve accessions and labels in `cv_predictions`, and record n, selected k/C/gamma and metrics in `cv_per_round`. Record the outer-refit selection in `selected_features`.

**Deliverable to Person 5:** five result rows, test predictions for all 8,146 training proteins once each, selected-feature records and `inner_scores`. These counts describe the intended coverage, not executed results. Do not tune against external predictions or evaluate benchmarking here.

**Similar code:** slides 39–41 define the outputs. SVM notebook section 1 provides `evaluate`; its single holdout demonstration is not the five-round project protocol.

## 6. Results and integration

We followed 03's reporting conventions and the practical requirement to report each metric as mean ± standard error across the five external folds. All tables, figures and observations below are pending.

### 6.1 Per-round results and mean ± SE

**Pending — Person 5; receives Person 4's completed CV results.** Display `cv_per_round` and summarize precision, recall, accuracy, MCC and F1 across the five rounds. Use the sample standard deviation (`ddof = 1`) and SE = SD / sqrt(5), matching 03. Keep summary columns `mean`, `std` and `stderr`, indexed by metric.

**Deliverable:** `table_dir / 6_1_cv_per_round.tsv`, `table_dir / 6_1_cv_summary.tsv` and `table_dir / 6_1_cv_predictions.tsv`. Save full precision and round only for display; label reported uncertainty as SE.

**Similar code:** 03 sections 5.4–5.5 already save per-round tables and calculate mean ± SE. The practical SVM notebook section 1 supplies individual metrics; slides 39–41 require the five-round summary.

### 6.2 Selected settings and feature records

**Pending — Person 5.** Show the selected n, k, C and gamma for each outer round, together with feature names and their outer-training importances. Export `selected_features`, `inner_scores` and the feature catalog. Describe variation across training sets without treating one ranked list as a biological finding.

**Deliverable:** `table_dir / 6_2_selected_features.tsv`, `table_dir / 6_2_inner_scores.tsv` and `table_dir / 6_2_feature_catalog.tsv`. The catalog must distinguish variable first-n composition from the fixed 40-residue hydropathy segment.

**Similar code:** slide 39 illustrates the settings table; slide 35 exposes importances. The two practical notebooks do not export these project tables. Follow the TSV convention in 03 section 5.

### 6.3 Plot external-fold metrics

**Pending — Person 5.** Plot the five metric means with SE and individual external-fold values, using consistent labels and colors. Save the figure as `figure_dir / 6_3_cv_metrics.pdf`, print its path and display it.

**Deliverable:** the external-fold metric figure. The points and error bars must come from the result tables; do not use illustrative values from slide 39 or outputs from either example dataset.

**Similar code:** 03 section 5.6 provides the mean ± SE plot. The practical notebooks show other plots, but not this five-round summary.

### 6.4 Observations and final integration

**Pending — Person 5.** Write brief observations about SP detection, parameter variation and selected features, using the actual external results. Read `../results/tables/03/5_4_cv_per_round.tsv` and `../results/tables/03/5_5_cv_summary.tsv` only for a descriptive comparison with the completed Von Heijne method; its training protocol differs from the SVM protocol.

Bring the five contributions together, retain the scientific order, and replace completed task instructions with explanations of what we actually did. Update README status/results when supported by real implementation and checks. Keep benchmarking reserved, and leave any unfinished tasks marked pending.

**Deliverable:** the integrated notebook and a short results-based discussion. Model internals, probability calibration, repeated random splits and all toy exercises are supporting examples rather than extra assigned analyses.

**Similar code:** the interpretation in 03 section 5.10 is a style reference. The practical examples explain their own proteins/datasets, so their observations cannot serve as our results.